# RAGAS Metrics

This notebook computes RAGAS metrics for some example Q/A pairs.

1. Build RAG Chain
2. Submit a RAG Query
3. Retrieve Context Docs
4. Faithfulness Score
5. Answer Relevancy Score
6. Context Precision
7. Context Recall
8. Combined Score
9. Summary

In [1]:
import logging

from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO)

# Vectorstore files were persisted here in notebook 1.vectorstore.ipynb
PERSIST_DIR = ".data/vectorstore"
COLLECTION_NAME = "arabidopsis_abstracts"
RAG_MODEL = "claude-sonnet-4-6"
EMB_MODEL = "all-MiniLM-L6-v2"

### 1. Build RAG Chain

In [2]:
from llm_knowledge_discovery.vectorstore import load_vectorstore
from llm_knowledge_discovery.rag import build_rag_chain

vectorstore = load_vectorstore(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

chain = build_rag_chain(
    vectorstore=vectorstore,
    model=RAG_MODEL,
    retrieval_k=10,
    rerank_k=5,
    temperature=0.5,
    max_tokens=4096
)

print(f"RAG Chain:\n{chain}")

/Users/dylanelliott/workspace/llm-knowledge-discovery/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Loading vectorstore: collection='arabidopsis_abstracts', persist_dir='.data/vectorstore'
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Vectorstore loaded from '.data/vectorstore'
INFO:llm_knowledge_discovery.rag.chain:[chain.py] RAG chain built: model='claude-sonnet-4-6', retrieval_k=10, rerank_k=5


RAG Chain:
first={
  context: RunnableLambda(lambda q: retrieve_and_rerank(q, vectorstore, retrieval_k, rerank_k))
           | RunnableLambda(_format_context),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You are a plant biology research assistant. Answer the user's question using ONLY the provided abstracts. If the abstracts do not contain enough information to answer the question, say so and do not include a references section. If you can answer the question, select only the abstracts you actually use, renumber them sequentially starting from [1], cite them inline by their new number (e.g. [1], [2]), and include a References section at the end listing only the abstracts you cited in that same sequential order, using their exact titles as they appear in the conte

### 2. Submit a RAG Query

In [3]:
query = "What are 2-3 single-gene edits that could result in repression of "\
    "SOC1 expression in Arabidopsis?"

# query = "What sort of single knock-out or over-expression strategies could "\
#     "potentially increase seed size and weight in Arabidopsis?"

rag_result = chain.invoke(query)

print(f"Query:\n{query}\n")
print(f"Answer:\n{rag_result.answer}\n")
print(f"References:\n{rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Query:
What are 2-3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis?

Answer:
Based on the provided abstracts, here are 2–3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis:

1. **Loss-of-function mutation of AtDIV1 (AtDIVARICATA1):** AtDIV1 is a MYB-like transcription factor that directly binds to the promoter region of SOC1 (as demonstrated by ChIP and EMSA analyses). Knockout or loss-of-function mutations of AtDIV1 (e.g., the *div1-1* or *div1-2* mutants) result in delayed flowering, consistent with reduced SOC1 activation [1].

2. **Loss-of-function mutation of ING2:** ING2 is a chromatin reader that associates with the NuA4 histone H4 acetyltransferase complex and is recruited to the chromatin of SOC1. ING2 is required for timely activation of SOC1 by modulating histone H4 acetylation (H4ac) levels at the SOC1 locus. A loss-of-function mutation in *ING2* would therefore be expected to reduce H4ac at SOC1 and re

### 3. Retrieve Context Docs

These are used explicitly for some scores.

In [4]:
from llm_knowledge_discovery.rag import retrieve_and_rerank

context_docs = retrieve_and_rerank(query, vectorstore)

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.20it/s]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents


### 4. Faithfulness Score

Does the answer make claims supported by the retrieved context? This metric can 
be used to detect hallucinations. 

In [5]:
from llm_knowledge_discovery.eval import score_faithfulness

faithfulness_result = score_faithfulness(
    query=query,
    answer=rag_result.answer,
    context_docs=context_docs,
    rag_model=RAG_MODEL,
    temperature=0,
    max_tokens=2048
)

INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.eval.faithfulness:[faithfulness.py] Extracted 21 claims from answer
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.a

In [6]:
dict(faithfulness_result)

{'query': 'What are 2-3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis?',
 'answer': 'Based on the provided abstracts, here are 2–3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis:\n\n1. **Loss-of-function mutation of AtDIV1 (AtDIVARICATA1):** AtDIV1 is a MYB-like transcription factor that directly binds to the promoter region of SOC1 (as demonstrated by ChIP and EMSA analyses). Knockout or loss-of-function mutations of AtDIV1 (e.g., the *div1-1* or *div1-2* mutants) result in delayed flowering, consistent with reduced SOC1 activation [1].\n\n2. **Loss-of-function mutation of ING2:** ING2 is a chromatin reader that associates with the NuA4 histone H4 acetyltransferase complex and is recruited to the chromatin of SOC1. ING2 is required for timely activation of SOC1 by modulating histone H4 acetylation (H4ac) levels at the SOC1 locus. A loss-of-function mutation in *ING2* would therefore be expected to reduce H4ac a

### 5. Answer Relevancy Score

Does the answer actually address what was asked?

In [7]:
from llm_knowledge_discovery.eval import score_answer_relevancy

answer_relevancy_result = score_answer_relevancy(
    query=query,
    answer=rag_result.answer,
    k=5,
    rag_model=RAG_MODEL,
    emb_model=EMB_MODEL,
    temperature=0,
    max_tokens=2048
)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.eval.answer_relevancy:[answer_relevancy.py] Generated 5 relevant queries from answer
INFO:llm_knowledge_discovery.eval.answer_relevancy:[answer_relevancy.py] Generated query [1]: Which single-gene mutations can repress SOC1 expression in Arabidopsis?
INFO:llm_knowledge_discovery.eval.answer_relevancy:[answer_relevancy.py] Generated query [2]: What is the role of AtDIV1 in regulating SOC1 and flowering time in Arabidopsis?
INFO:llm_knowledge_discovery.eval.answer_relevancy:[answer_relevancy.py] Generated query [3]: How does ING2 regulate SOC1 expression through histone acetylation?
INFO:llm_knowledge_discovery.eval.answer_relevancy:[answer_relevancy.py] Generated query [4]: What genes when knocked ou

In [8]:
dict(answer_relevancy_result)

{'query': 'What are 2-3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis?',
 'answer': 'Based on the provided abstracts, here are 2–3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis:\n\n1. **Loss-of-function mutation of AtDIV1 (AtDIVARICATA1):** AtDIV1 is a MYB-like transcription factor that directly binds to the promoter region of SOC1 (as demonstrated by ChIP and EMSA analyses). Knockout or loss-of-function mutations of AtDIV1 (e.g., the *div1-1* or *div1-2* mutants) result in delayed flowering, consistent with reduced SOC1 activation [1].\n\n2. **Loss-of-function mutation of ING2:** ING2 is a chromatin reader that associates with the NuA4 histone H4 acetyltransferase complex and is recruited to the chromatin of SOC1. ING2 is required for timely activation of SOC1 by modulating histone H4 acetylation (H4ac) levels at the SOC1 locus. A loss-of-function mutation in *ING2* would therefore be expected to reduce H4ac a

### 6. Context Precision Score

Was the retrieved context relevant for answering the question?

In [9]:
from llm_knowledge_discovery.eval import score_context_precision

context_precision_result = score_context_precision(
    query=query,
    answer=rag_result.answer,
    context_docs=context_docs,
    rag_model=RAG_MODEL,
    temperature=0,
    max_tokens=2048
)

INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.eval.context_precision:[context_precision.py] 3/5 context documents are relevant
INFO:llm_knowledge_discovery.eval.context_precision:[context_precision.py] Context Precision score: 0.87


In [10]:
dict(context_precision_result)

{'query': 'What are 2-3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis?',
 'answer': 'Based on the provided abstracts, here are 2–3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis:\n\n1. **Loss-of-function mutation of AtDIV1 (AtDIVARICATA1):** AtDIV1 is a MYB-like transcription factor that directly binds to the promoter region of SOC1 (as demonstrated by ChIP and EMSA analyses). Knockout or loss-of-function mutations of AtDIV1 (e.g., the *div1-1* or *div1-2* mutants) result in delayed flowering, consistent with reduced SOC1 activation [1].\n\n2. **Loss-of-function mutation of ING2:** ING2 is a chromatin reader that associates with the NuA4 histone H4 acetyltransferase complex and is recruited to the chromatin of SOC1. ING2 is required for timely activation of SOC1 by modulating histone H4 acetylation (H4ac) levels at the SOC1 locus. A loss-of-function mutation in *ING2* would therefore be expected to reduce H4ac a

### 7. Context Recall

TODO: Need to create gold-standard QA pair dataset first. 

### 8. Combined Score

In [11]:
combined_score = (
    faithfulness_result.score + \
    answer_relevancy_result.score + \
    context_precision_result.score
) / 3

### 9. Summary

In [12]:
print(f"Query: {query}")
print(f"Answer: {rag_result.answer}")
print(f"References: {rag_result.references}")
print(f"Faithfulness Score: {faithfulness_result.score:.04f}")
print(f"Answer Relevancy Score: {answer_relevancy_result.score:.04f}")
print(f"Context Precision Score: {context_precision_result.score:.04f}")
print(f"Combined Score: {combined_score:.04f}")

Query: What are 2-3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis?
Answer: Based on the provided abstracts, here are 2–3 single-gene edits that could result in repression of SOC1 expression in Arabidopsis:

1. **Loss-of-function mutation of AtDIV1 (AtDIVARICATA1):** AtDIV1 is a MYB-like transcription factor that directly binds to the promoter region of SOC1 (as demonstrated by ChIP and EMSA analyses). Knockout or loss-of-function mutations of AtDIV1 (e.g., the *div1-1* or *div1-2* mutants) result in delayed flowering, consistent with reduced SOC1 activation [1].

2. **Loss-of-function mutation of ING2:** ING2 is a chromatin reader that associates with the NuA4 histone H4 acetyltransferase complex and is recruited to the chromatin of SOC1. ING2 is required for timely activation of SOC1 by modulating histone H4 acetylation (H4ac) levels at the SOC1 locus. A loss-of-function mutation in *ING2* would therefore be expected to reduce H4ac at SOC1 and rep